In [4]:
# ============================================================
# Confidence Score Distribution
# Panel A: GMM-based distribution (single dataset, Original)
# Panel B: Raincloud plot across datasets
# Panel C: Stacked bar chart of confidence tiers per dataset
# ============================================================
library(ggplot2)
library(dplyr)
library(tidyr)
library(patchwork)
library(ggdist)
library(scales)

gmm_breaks <- c(0.00, 0.45, 0.77, 0.97)
gmm_labels <- c("Low (0.00–0.45)", "Medium (0.45–0.77)", "High (0.77–0.97)")
gmm_colors <- c("Low (0.00–0.45)"    = "#0571b0",
                "Medium (0.45–0.77)" = "#92c5de",
                "High (0.77–0.97)"   = "#ca0020")

paths <- list(
  "Original"     = "Original_Validation_Merged.csv",
  "Random"       = "Random_Validation_Merged.csv",
  "50% Shuffled" = "Partial_Validation_Merged.csv"
)

orig_df <- read.csv(paths[["Original"]], stringsAsFactors = FALSE)
orig_df$Final_Confidence <- suppressWarnings(as.numeric(orig_df$Final_Confidence))
orig_df <- orig_df[!is.na(orig_df$Final_Confidence), ]

orig_df$Tier <- cut(
  orig_df$Final_Confidence,
  breaks = gmm_breaks,
  labels = gmm_labels,
  include.lowest = TRUE,
  right = FALSE
)

conf_data <- bind_rows(
  lapply(names(paths), function(nm) {
    df <- read.csv(paths[[nm]], stringsAsFactors = FALSE)
    df$Final_Confidence <- suppressWarnings(as.numeric(df$Final_Confidence))
    data.frame(
      Dataset    = nm,
      Confidence = df$Final_Confidence[!is.na(df$Final_Confidence)]
    )
  })
) %>%
  mutate(
    Dataset = factor(Dataset, levels = c("Random", "50% Shuffled", "Original")),
    Tier    = cut(Confidence,
                  breaks = gmm_breaks,
                  labels = gmm_labels,
                  include.lowest = TRUE,
                  right = FALSE)
  )

dataset_pal <- c(
  "Random"       = "#92c5de",
  "50% Shuffled" = "#0571b0",
  "Original"     = "#ca0020"
)

theme_nature <- function(base_size = 10) {
  theme_classic(base_size = base_size, base_family = "Helvetica") +
    theme(
      panel.border       = element_rect(colour = "black", fill = NA, linewidth = 0.6),
      panel.grid.major.y = element_line(colour = "grey92", linewidth = 0.35),
      panel.grid.minor   = element_blank(),
      axis.line          = element_blank(),
      axis.ticks         = element_line(colour = "black", linewidth = 0.45),
      axis.ticks.length  = unit(3, "pt"),
      axis.title         = element_text(face = "bold", size = base_size),
      axis.text          = element_text(colour = "black", size = base_size - 1),
      legend.title       = element_text(face = "bold", size = base_size - 1),
      legend.text        = element_text(size = base_size - 2),
      legend.key         = element_blank(),
      legend.background  = element_blank(),
      plot.title         = element_text(face = "bold", size = base_size + 1, hjust = 0),
      plot.subtitle      = element_text(size = base_size - 2, colour = "grey45", hjust = 0),
      strip.background   = element_rect(fill = "grey96", colour = "black", linewidth = 0.5),
      strip.text         = element_text(face = "bold", size = base_size - 1),
      plot.margin        = margin(8, 16, 8, 8)
    )
}

# ============================================================
# PANEL A — GMM Confidence Distribution (Original dataset)
# ============================================================

thresholds <- gmm_breaks[c(-1, -length(gmm_breaks))]

band_mids  <- c(0.225, 0.610, 0.870)
band_annot <- data.frame(
  x     = band_mids,
  label = c("Low\n0.00–0.45",
            "Medium\n0.45–0.77",
            "High\n0.77–0.97")
)

band_df_A <- data.frame(
  ymin  = c(0.00, 0.45, 0.77),
  ymax  = c(0.45, 0.77, 1.00),
  bfill = c("#fce8e8", "#fefae7", "#eaf4ea")
)

orig_colour <- "#ca0020"

pA <- ggplot(orig_df, aes(x = Final_Confidence)) +
  geom_rect(data = band_df_A, inherit.aes = FALSE,
            aes(xmin = ymin, xmax = ymax, ymin = -Inf, ymax = Inf),
            fill = band_df_A$bfill, alpha = 0.6) +
  geom_histogram(
    aes(y = after_stat(count)),
    bins      = 30,
    fill      = orig_colour,
    colour    = "black",
    linewidth = 0.35,
    alpha     = 0.80
  ) +
  geom_density(
    aes(y = after_stat(density) * nrow(orig_df) * (0.97 / 30)),
    colour    = "grey20",
    linewidth = 0.5,
    fill      = NA
  ) +

  geom_vline(
    xintercept = thresholds,
    linetype   = "dashed",
    colour     = "grey40",
    linewidth  = 0.6
  ) +

  annotate(
    "text",
    x      = band_annot$x,
    y      = Inf,
    label  = band_annot$label,
    vjust  = 1.25,
    size   = 2.5,
    colour = "grey30",
    fontface = "italic",
    lineheight = 1.1
  ) +
  scale_x_continuous(
    limits = c(0, 1),
    breaks = seq(0, 1, 0.2),
    expand = expansion(mult = c(0, 0.01))
  ) +
  scale_y_continuous(
    expand = expansion(mult = c(0, 0.06))
  ) +
  labs(
    title    = "a",
    subtitle = "GMM-based confidence distribution (Original dataset, k=3)",
    x        = "Confidence Score",
    y        = "Frequency"
  ) +
  theme_nature()

# ============================================================
# PANEL B — Raincloud (dodged, all 3 datasets)
# ============================================================

band_df <- data.frame(
  ymin  = c(0.00, 0.45, 0.77),
  ymax  = c(0.45, 0.77, 0.97),
  bfill = c("#fce8e8", "#fefae7", "#eaf4ea"),
  label = c("Low", "Medium", "High"),
  label_y = c(0.225, 0.610, 0.870)
)

pB <- ggplot(conf_data,
             aes(x = Dataset, y = Confidence, fill = Dataset, colour = Dataset)) +
  geom_rect(data = band_df, inherit.aes = FALSE,
            aes(xmin = -Inf, xmax = Inf, ymin = ymin, ymax = ymax),
            fill = band_df$bfill, alpha = 0.5) +
  geom_hline(yintercept = thresholds,
             linetype = "dashed", colour = "grey55", linewidth = 0.5) +
  stat_halfeye(adjust = 0.9, width = 0.5, .width = 0,
               justification = -0.18, point_colour = NA, alpha = 0.75,
               position = position_dodge(width = 0.8)) +
  geom_boxplot(outlier.shape = NA, width = 0.15, linewidth = 0.5,
               colour = "black", alpha = 0.6,
               position = position_dodge(width = 0.8)) +
  geom_point(aes(x = as.numeric(Dataset) - 0.22),
             position = position_jitter(width = 0.06, seed = 42),
             size = 0.9, alpha = 0.35, shape = 16) +
  annotate("text",
           x = 3.62, y = band_df$label_y,
           label = band_df$label,
           hjust = 0, size = 2.8, colour = "grey40", fontface = "italic") +
  scale_fill_manual(values = dataset_pal, name = "Dataset") +
  scale_colour_manual(values = dataset_pal, name = "Dataset") +
  scale_y_continuous(limits = c(0, 1.02),
                     breaks = seq(0, 1, 0.2),
                     expand = expansion(mult = c(0.01, 0.03))) +
  coord_cartesian(xlim = c(0.4, 3.85), clip = "off") +
  labs(
    title    = "b",
    subtitle = "Distribution of confidence scores across datasets",
    x        = "Dataset",
    y        = "Confidence Score"
  ) +
  theme_nature() +
  theme(legend.position = "none")

# ============================================================
# PANEL C — Grouped bar chart of confidence tiers
# ============================================================

tier_counts <- conf_data %>%
  filter(!is.na(Tier)) %>%
  count(Dataset, Tier) %>%
  group_by(Dataset) %>%
  mutate(pct = round(n / sum(n) * 100)) %>%
  ungroup() %>%
  mutate(Dataset = factor(Dataset, levels = c("Random", "50% Shuffled", "Original")))

pC <- ggplot(tier_counts, aes(x = Tier, y = n, fill = Dataset)) +
  geom_col(position = position_dodge(width = 0.75), width = 0.68,
           colour = "white", linewidth = 0.4) +
  geom_text(aes(label = paste0(n, " (", pct, "%)")),
            position = position_dodge(width = 0.75),
            vjust = -0.45, size = 2.3, fontface = "bold", colour = "black") +
  scale_fill_manual(values = dataset_pal, name = "Dataset") +
  scale_y_continuous(expand = expansion(mult = c(0, 0.20)),
                     breaks = pretty_breaks(5)) +
  labs(
    title    = "c",
    subtitle = "Confidence tier distribution across datasets",
    x        = "Confidence Interval",
    y        = "Number of Gene Sets"
  ) +
  theme_nature() +
  theme(panel.grid.major.x = element_blank())

# ============================================================
# SAVE
# ============================================================
fig <- (pA | pB | pC) +
  plot_layout(guides = "collect") &
  theme(legend.position = "bottom",
        legend.direction = "horizontal")

ggsave("confidence_analysis_figurev2.pdf",
       fig, width = 21, height = 6, dpi = 300)
ggsave("confidence_analysis_figurev2.png",
       fig, width = 21, height = 6, dpi = 300)

cat("Saved confidence_analysis_figurev2.pdf/.png\n")

Warning message:
“Removed 2 rows containing missing values or values outside the scale range
(`geom_bar()`).”
Warning message:
“Removed 2 rows containing missing values or values outside the scale range
(`geom_slabinterval()`).”
Warning message:
“Removed 46 rows containing missing values or values outside the scale range
(`geom_point()`).”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for '0.00–0.45' in 'mbcsToSbcs': - substituted for – (U+2013)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for '0.45–0.77' in 'mbcsToSbcs': - substituted for – (U+2013)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for '0.77–0.97' in 'mbcsToSbcs': - substituted for – (U+2013)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Low (0.00–0.45)' in 'mbcsToSbcs': - substituted for – (U+2013)”
Warning message in grid.Call.graphics(C_text, as.graphic

Saved confidence_analysis_figurev2.pdf/.png
